In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

from openpyxl import load_workbook

In [13]:
# 파일 불러오기
bike = pd.read_csv('bike_merged.csv', encoding='utf-8-sig')

In [3]:
# nunique 확인
def uni_df(dataframe):
    for col in dataframe:
        print(col,dataframe[col].unique())
        print(col,dataframe[col].nunique())

In [4]:
uni_df(bike)

대여일시 ['2024-01-01 00:04:10' '2024-01-01 00:00:10' '2024-01-01 00:03:13' ...
 '2024-12-31 23:46:21' '2024-12-31 23:46:26' '2024-12-31 23:19:31']
대여일시 19455927
대여 대여소명 ['동서울농협 앞' '신대방삼거리' '군자역 7번출구 베스트샵 앞' ... '하계동 중평어린이공원 앞' '정곡나들목'
 '노해근린공원내']
대여 대여소명 2767


In [4]:
# 대여소명만 추출해서 텍스트 파일로 저장
unique_stations = bike['대여 대여소명'].drop_duplicates().sort_values()
stations_path = "unique_stations.txt"
unique_stations.to_csv(stations_path, index=False, header=False)

In [14]:
# 엑셀 파일 열기
wb = load_workbook("station_name.xlsx")
ws = wb.active

# 정확히 B6:C 끝까지 필요한 값만 수집
data = []
for row in ws.iter_rows(min_row=6, min_col=1, max_col=3, values_only=True):
    if row[0] == '대여소명' or row[0] is None:
        continue
    data.append(row)

# DataFrame으로 변환
station_map = pd.DataFrame(data, columns=["stat_num","stat", "gu"])

D:\mini_proj1\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
D:\mini_proj1\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [11]:
station_map.head()

,stat_num,stat,gu
0,00301,경복궁역 7번출구 앞,종로구
1,00302,경복궁역 4번출구 뒤,종로구
2,00303,광화문역 1번출구 앞,종로구
3,00305,종로구청 옆,종로구
4,00307,서울역사박물관 앞,종로구


In [17]:
bike.head()

,대여일시,대여 대여소명,대여 대여소번호
0,2024-01-01,동서울농협 앞,4804
1,2024-01-01,신대방삼거리,4169
2,2024-01-01,군자역 7번출구 베스트샵 앞,540
3,2024-01-01,용문사 버스정류장,1139
4,2024-01-01,동묘앞역 6번출구,3416


In [15]:
# 정수로 변환
station_map["stat_num"] = station_map["stat_num"].astype(int)

# 5자리 문자열로 포맷
station_map["stat_num"] = station_map["stat_num"].apply(lambda x: f"{x:05d}")

In [18]:
# 정수로 변환
bike["대여 대여소번호"] = bike["대여 대여소번호"].astype(int)

# 5자리 문자열로 포맷
bike["대여 대여소번호"] = bike["대여 대여소번호"].apply(lambda x: f"{x:05d}")

In [19]:
bike = bike[bike["대여 대여소번호"] != "\\N"]

In [20]:
# 3. '대여 대여소번호' 기준으로 자치구 병합
bike = bike.merge(station_map, left_on='대여 대여소번호', right_on='stat_num', how='left')

In [21]:
bike.head()

,대여일시,대여 대여소명,대여 대여소번호,stat_num,stat,gu
0,2024-01-01,동서울농협 앞,04804,04804,동서울농협 앞,중랑구
1,2024-01-01,신대방삼거리,04169,04169,신대방삼거리,동작구
2,2024-01-01,군자역 7번출구 베스트샵 앞,00540,00540,군자역 7번출구 베스트샵 앞,광진구
3,2024-01-01,용문사 버스정류장,01139,01139,용문사 버스정류장,강서구
4,2024-01-01,동묘앞역 6번출구,03416,03416,동묘앞역 6번출구,종로구


In [26]:
# 4. 불필요한 컬럼 제거
bike.drop(columns=['대여 대여소명','stat_num','stat'], inplace=True)

In [28]:
bike = bike.dropna(subset=['gu'])

In [32]:
bike.shape

(43880014, 3)

In [29]:
bike.isnull().sum()

대여일시        0
대여 대여소번호    0
gu          0
dtype: int64

In [30]:
bike = bike.rename(columns={'gu': '자치구'})

In [34]:
bike_usage = (
    bike.groupby(['대여일시', '자치구'])
    .size()
    .reset_index(name='총사용수')
)

In [35]:
bike_usage.head()

,대여일시,자치구,총사용수
0,2024-01-01,강남구,1319
1,2024-01-01,강동구,1855
2,2024-01-01,강북구,731
3,2024-01-01,강서구,5668
4,2024-01-01,관악구,1438


In [36]:
# 5. 결과 저장
bike_usage.to_csv("bike_with_gu.csv", index=False)

In [2]:
bike_usage.shape

NameError: name 'bike_usage' is not defined

In [1]:
bike.isnull().sum()

NameError: name 'bike' is not defined